# Kinetic isotope effects with PyQuiver

A **kinetic isotope effect (KIE)** compares how fast a reaction goes for two versions of the same molecule that differ only in their isotopes, for example a normal molecule versus one with a single ¹²C replaced by ¹³C. The heavier isotope vibrates a little more slowly, which nudges the activation barrier and therefore the rate; that small ratio is a classic probe of reaction mechanism.

PyQuiver computes KIEs from the vibrational frequencies of the **reactant** (the *ground state*) and the **transition state** (the high point along the reaction path), using the Bigeleisen-Mayer equation. For each of those two structures you supply a frequency calculation in the form of its Cartesian Hessian, plus a short configuration listing the isotopic substitutions you want.

The examples below use the Claisen rearrangement files shipped with PyQuiver.

In [1]:
import numpy as np
import pandas as pd
import cctk
from IPython.display import display

from pyquiver import KIE_Calculation, System, Config, Isotopologue, batch
from pyquiver.constants import DEFAULT_MASSES, REPLACEMENTS
from pyquiver.kie import calculate_rpfr, partition_components

## A simple calculation

A calculation needs three things: a **configuration** (what to substitute, and the conditions), a **ground-state** file, and a **transition-state** file. The easiest way to make the configuration is `Config.from_dict`.

Its `isotopologues` argument is a dictionary. Each key is a name you choose, and its value lists the substitutions to make as `(ground_state_atom, transition_state_atom, replacement)` tuples. Atom numbers start at 1 and follow the numbering in your output files; `replacement` is an isotope label from `weights.dat` such as `"13C"`, `"2D"`, or `"17O"`. Listing **more than one tuple substitutes several atoms at once**, as in `"H/D"` below, which deuterates two hydrogens together. (You can also give a plain number instead of a label to use an unusual mass, e.g. `(4, 4, 5000.0)`.)

The remaining settings are the `temperature` in kelvin, an empirical `scaling` factor applied to the frequencies, and `imag_threshold` in cm⁻¹ (a small imaginary frequency below this cutoff is ignored rather than treated as the reaction mode).

In [2]:
config = Config.from_dict(
    isotopologues={
        "C1":  [(1, 1, "13C")],                 # single substitution
        "O3":  [(3, 3, "17O")],
        "H/D": [(7, 7, "2D"), (8, 8, "2D")],    # double substitution: two atoms at once
    },
    temperature=393, scaling=0.9614, imag_threshold=50,
)

calc = KIE_Calculation(config, "../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out",
                       style="gaussian")
print(calc)


=== PyQuiver Analysis ===
Isotopologue                                              uncorrected      Wigner     inverted parabola
                                                              KIE           KIE              KIE
Isotopologue         C1                                      1.0127        1.0144          1.0147      
Isotopologue         O3                                      1.0188        1.0204          1.0206      
Isotopologue        H/D                                      0.9548        0.9563          0.9566      

Absolute KIEs are given.


The same calculation works with output from other programs; you only change `style`. PyQuiver reads Gaussian output files, ORCA `.hess` files, and its own compact `.qin` format, and the configuration stays exactly the same:

In [3]:
orca   = KIE_Calculation(config, "../tutorial/orca/claisen_gs_freq.hess", "../tutorial/orca/claisen_ts_freq.hess", style="orca")
native = KIE_Calculation(config, "../tutorial/pyquiver/claisen_gs.qin", "../tutorial/pyquiver/claisen_ts.qin", style="pyquiver")

print(f"Gaussian C1: {calc.results['C1'].uncorrected:.4f}")
print(f"ORCA     C1: {orca.results['C1'].uncorrected:.4f}")
print(f"native   C1: {native.results['C1'].uncorrected:.4f}")

Gaussian C1: 1.0127
ORCA     C1: 1.0130
native   C1: 1.0127


## Getting the results as data

Printing the calculation gives a formatted table, but usually you want the numbers themselves. Each isotopologue has three KIE values: the **uncorrected** one and two tunnelling-corrected ones, **Wigner** and **Bell** (both explained later). Look one up by name, or get the whole set as a table with `to_dataframe()` (which needs `pandas`):

In [4]:
c1 = calc.results["C1"]
print(f"C1 uncorrected: {c1.uncorrected:.4f}, Wigner: {c1.wigner:.4f}, inverted parabola: {c1.inverted_parabola:.4f}")

display(calc.to_dataframe())

C1 uncorrected: 1.0127, Wigner: 1.0144, inverted parabola: 1.0147


,name,uncorrected,wigner,inverted_parabola
0,C1,1.012746,1.014443,1.014743
1,O3,1.018842,1.020365,1.020635
2,H/D,0.954837,0.956311,0.956572


## Reusing parsed files

Every time you pass a filename, PyQuiver opens and re-parses that file. If you plan to run several calculations on the same structures, read each file once into a `System` object and reuse it. For example, to see how the KIE changes with temperature:

In [5]:
gs = System("../tutorial/gaussian/claisen_gs.out")
ts = System("../tutorial/gaussian/claisen_ts.out")

for T in (298, 350, 393):
    cfg = Config.from_dict(isotopologues={"C1": [(1, 1, "13C")]},
                           temperature=T, scaling=0.9614, imag_threshold=50)
    print(f"{T} K: C1 KIE = {KIE_Calculation(cfg, gs, ts).results['C1'].uncorrected:.4f}")

298 K: C1 KIE = 1.0144
350 K: C1 KIE = 1.0134
393 K: C1 KIE = 1.0127


## Running many files at once

Often you have many ground-state/transition-state pairs to run with the same configuration (for example, the same reaction at several levels of theory). `batch` runs them all and returns one combined table. You can describe the pairs in three equivalent ways.

**1. A dictionary**, where each key is a label and each value is the `(ground_state, transition_state)` pair:

In [6]:
results = batch(config, {
    "run1": ("../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
    "run2": ("../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
})
display(results.to_dataframe())

,label,name,uncorrected,wigner,inverted_parabola
0,run1,C1,1.012746,1.014443,1.014743
1,run1,O3,1.018842,1.020365,1.020635
2,run1,H/D,0.954837,0.956311,0.956572
3,run2,C1,1.012746,1.014443,1.014743
4,run2,O3,1.018842,1.020365,1.020635
5,run2,H/D,0.954837,0.956311,0.956572


**2. A list of `(label, ground_state, transition_state)` rows**, if you would rather not write a dictionary:

In [7]:
results = batch(config, [
    ("run1", "../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
    ("run2", "../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
])
display(results.to_dataframe())

,label,name,uncorrected,wigner,inverted_parabola
0,run1,C1,1.012746,1.014443,1.014743
1,run1,O3,1.018842,1.020365,1.020635
2,run1,H/D,0.954837,0.956311,0.956572
3,run2,C1,1.012746,1.014443,1.014743
4,run2,O3,1.018842,1.020365,1.020635
5,run2,H/D,0.954837,0.956311,0.956572


**3. A list of `(ground_state, transition_state)` pairs** with no labels, in which case PyQuiver names each row after its ground-state file:

In [8]:
results = batch(config, [
    ("../tutorial/gaussian/claisen_gs.out", "../tutorial/gaussian/claisen_ts.out"),
])
display(results.to_dataframe())

,label,name,uncorrected,wigner,inverted_parabola
0,claisen_gs,C1,1.012746,1.014443,1.014743
1,claisen_gs,O3,1.018842,1.020365,1.020635
2,claisen_gs,H/D,0.954837,0.956311,0.956572


## Tunnelling corrections

Near the transition state, light nuclei can "tunnel" through the energy barrier instead of passing over it, which speeds the reaction up (most noticeably for hydrogen). PyQuiver reports every KIE three ways: **uncorrected** (no tunnelling) plus two common corrections, **Wigner** and **Bell** (the "inverted parabola").

The **Bell inverted parabola is the default**; it is the corrected value you would normally quote, and it is already in the table above. A third correction, **Skodje-Truhlar**, is worth using when the imaginary frequency is very large and the inverted-parabola picture breaks down; because it also needs the reaction's barrier height, it is computed on request rather than reported automatically.

Each formula below gives a single-mode coefficient $\kappa$, and the factor applied to a KIE is the light/heavy ratio $\kappa_{light}/\kappa_{heavy}$. For an imaginary mode of magnitude $\tilde\nu^\ddag$, write $u = {h c\, \tilde\nu^\ddag \over k_B T}$.

### Wigner

$$ \kappa = 1 + {u^2 \over 24} $$

### Bell (inverted parabola)

$$ \kappa = {u/2 \over \sin(u/2)} $$

### Skodje-Truhlar

An alternative (*J. Phys. Chem.* **1981**, 85, 624-626) that can be more applicable for very large imaginary frequencies. With $\alpha = {2\pi \over h\nu^\ddag}$, $\beta = {1 \over k_B T}$, and the barrier height in the exothermic direction $V = V_{TS} - \max(V_{SM}, V_{PR})$:

small $\nu^\ddag$ / high $T$ ($\alpha > \beta$):
$$ \kappa = {\beta\pi/\alpha \over \sin(\beta\pi/\alpha)} - {\beta \over \alpha - \beta}\,e^{(\beta-\alpha)V} $$

large $\nu^\ddag$ / low $T$ ($\alpha < \beta$):
$$ \kappa = {\beta \over \beta-\alpha}\left(e^{(\beta-\alpha)V} - 1\right) $$

and $\kappa = {\beta\pi/\alpha \over \sin(\beta\pi/\alpha)}$ when $\alpha = \beta$.

Skodje-Truhlar needs electronic energies, read here from the Gaussian outputs with `cctk`:

In [9]:
def energy(path):
    ensemble = cctk.GaussianFile.read_file(path).ensemble
    return ensemble.get_properties_dict(ensemble.molecules[-1])["energy"]

E_reactant = energy("../tutorial/gaussian/claisen_gs.out")
E_ts = energy("../tutorial/gaussian/claisen_ts.out")

# energies in hartree; product taken equal to the reactant here
calc.skodje_truhlar(reactant_energy=E_reactant, product_energy=E_reactant, ts_energy=E_ts)

OrderedDict([('C1', np.float64(1.0147431211866438)),
             ('O3', np.float64(1.0206350979560215)),
             ('H/D', np.float64(0.956572091397035))])

## Under the hood: partition function ratios

This section is optional. Internally a KIE is built from **reduced isotopic partition function ratios** (RPFRs), one for the ground state and one for the transition state, each comparing the light and heavy isotopologue. To look inside, build the light and heavy `Isotopologue` objects yourself and call `calculate_rpfr`:

In [10]:
gs_masses = [DEFAULT_MASSES[z] for z in gs.atomic_numbers]
heavy_masses = list(gs_masses)
heavy_masses[0] = REPLACEMENTS["13C"]        # 13C at atom 1

light = Isotopologue("light", gs, gs_masses)
heavy = Isotopologue("heavy", gs, heavy_masses)

rpfr, imag_ratio, light_freqs, heavy_freqs = calculate_rpfr((light, heavy), 50.0, 0.9614, 393)
print(f"ground-state RPFR = {rpfr:.4f}")

ground-state RPFR = 1.0872


You can also see how each vibrational mode contributes to that ratio. `partition_components` returns, for every mode, its frequency (product), excitation, and zero-point factors; their product is the mode's contribution to the RPFR:

In [11]:
components = partition_components(heavy_freqs, light_freqs, 393)
modes = pd.DataFrame(components, columns=["product", "excitation", "ZPE"])
modes["contribution"] = modes.prod(axis=1)
display(modes.head())

,product,excitation,ZPE,contribution
0,1.008401,0.992653,0.998967,0.999957
1,1.000033,0.999976,0.999990,0.999999
2,1.005613,0.996045,0.998168,0.999801
3,1.001848,0.998863,0.999168,0.999877
4,1.011760,0.993243,0.994135,0.999030
